<a href="https://colab.research.google.com/github/kimshy22/AI-Battery-SOH-Monitoring/blob/main/hybrid1D_CNN_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1: Install & Imports
import numpy as np
import pandas as pd
from scipy import interpolate
import joblib
import json
import tensorflow as tf
from tensorflow.keras.models import model_from_json
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

print(f"TensorFlow version: {tf.__version__}")

TensorFlow version: 2.20.0


In [ ]:
# Cell 2: Universal Processing Function
VOLTAGE_GRID = np.linspace(3.90, 3.80, 100)

def process_and_normalize(df, nominal_capacity):
    """
    Processes raw battery discharge data into normalized, interpolated
    difference-curve features.

    Args:
        df (pd.DataFrame): Raw data with columns:
                           [cycle_id, type, time_s, voltage_v, current_a, true_soh]
        nominal_capacity (float): Nominal capacity in Ah (2.0 for NASA, 2.5 for physical).

    Returns:
        dict: {cycle_id: {'curve': np.array(100,), 'soh': float}}
              A dictionary mapping each cycle to its interpolated,
              normalized partial capacity curve and its true SOH label.
    """

    # --- Step 1: Filter for discharge cycles only ---
    # Charge and impedance rows carry no useful discharge curve shape information.
    if 'type' in df.columns:
        df = df[df['type'] == 'discharge'].copy()

    processed_cycles = {}

    for cycle_id, cycle_df in df.groupby('cycle_id'):

        # --- Step 2: Extract the 3.90V to 3.80V partial window ---
        # This simulates the voltage window most commonly observed in
        # real e-bike partial discharge events.
        window = cycle_df[
            (cycle_df['voltage_v'] >= 3.80) &
            (cycle_df['voltage_v'] <= 3.90)
        ].copy().sort_values('time_s')

        # Skip cycles that don't fully cover the window
        if window['voltage_v'].max() < 3.89 or window['voltage_v'].min() > 3.81:
            print(f"  [SKIP] Cycle {cycle_id}: does not cover 3.90V-3.80V window.")
            continue

        if len(window) < 5:
            print(f"  [SKIP] Cycle {cycle_id}: insufficient data points ({len(window)}).")
            continue

        # --- Step 3: Coulomb Counting (Trapezoidal Integration) ---
        # ΔQ = ∫I dt, computed cumulatively so we get Q as a function of time.
        # Current is negative during discharge in many datasets; take absolute value.
        time   = window['time_s'].values
        current = np.abs(window['current_a'].values)
        voltage = window['voltage_v'].values

        # Cumulative trapezoidal integration → Q in Ampere-seconds, convert to Ah
        delta_q_cumulative = np.cumsum(
            np.trapezoid(
                np.column_stack([current[:-1], current[1:]]),
                axis=1
            ) * np.diff(time)
        ) / 3600.0

        # Align voltage to cumulative Q (drop first voltage point to match diff length)
        voltage_aligned = voltage[1:]
        delta_q_aligned  = delta_q_cumulative

        # --- Step 4: Normalize by nominal capacity ---
        # Dividing by nominal_capacity (Ah) makes ΔQ a dimensionless percentage.
        # This is the key operation enabling Sim-to-Real transfer:
        # NASA (2.0Ah) and Physical (2.5Ah) curves now exist in the same feature space.
        delta_q_normalized = delta_q_aligned / nominal_capacity

        # --- Step 5: Interpolate onto fixed 100-point voltage grid ---
        # Voltage drops during discharge, so we reverse both arrays so that
        # voltage is monotonically increasing — required by scipy's interpolator.
        voltage_reversed = voltage_aligned[::-1]
        dq_reversed      = delta_q_normalized[::-1]

        # Remove duplicate voltage values that would break interpolation
        unique_mask = np.diff(voltage_reversed, prepend=-np.inf) > 0
        voltage_reversed = voltage_reversed[unique_mask]
        dq_reversed      = dq_reversed[unique_mask]

        # Ensure the window actually spans the grid before interpolating
        # The `fill_value='extrapolate'` in interp1d will handle cases where the voltage_reversed
        # range is slightly narrower than VOLTAGE_GRID, so this strict check is removed.
        # if voltage_reversed.min() > VOLTAGE_GRID.min() or \
        #    voltage_reversed.max() < VOLTAGE_GRID.max():
        #     print(f"  [SKIP] Cycle {cycle_id}: voltage span too narrow for grid.")
        #     continue

        interp_func   = interpolate.interp1d(voltage_reversed, dq_reversed,
                                              kind='linear', fill_value='extrapolate')
        curve_on_grid = interp_func(VOLTAGE_GRID)

        # Extract the SOH label for this cycle (take the mode/first value)
        soh = cycle_df['true_soh'].iloc[0]

        processed_cycles[cycle_id] = {'curve': curve_on_grid, 'soh': soh}

    print(f"\n[INFO] Successfully processed {len(processed_cycles)} cycles.")
    return processed_cycles

In [ ]:
# Cell: Mount Google Drive and Load NASA Data

from google.colab import drive
import pandas as pd

# --- Mount Drive ---
drive.mount('/content/drive')

NASA_DATA_PATH = '/content/drive/MyDrive/battery_combined_data/battery data/All_Batteries_Combined.csv'

nasa_df = pd.read_csv(NASA_DATA_PATH)
print("Loaded. Shape:", nasa_df.shape)
nasa_df.head()

Mounted at /content/drive
Loaded. Shape: (185721, 7)


,Cycle,Time,Voltage,Current,Temperature,Capacity,BatteryID
0,2,0.000,4.191492,-0.004902,24.330034,1.856487,B0005
1,2,16.781,4.190749,-0.001478,24.325993,1.856487,B0005
2,2,35.703,3.974871,-2.012528,24.389085,1.856487,B0005
3,2,53.781,3.951717,-2.013979,24.544752,1.856487,B0005
4,2,71.922,3.934352,-2.011144,24.731385,1.856487,B0005


In [ ]:
# Cell 3: NASA Pre-Training Data Preparation

# Rename columns to match the universal processing function's expectations
nasa_df = nasa_df.rename(columns={'Cycle': 'cycle_id', 'Time': 'time_s', 'Voltage': 'voltage_v', 'Current': 'current_a', 'Capacity': 'true_soh'})

# --- Process through universal pipeline ---
nasa_cycles = process_and_normalize(nasa_df, nominal_capacity=2.0)

# --- Build NASA Golden Baseline from cycles 5-15 ---
# Average across these cycles to smooth out single-cycle noise.
baseline_cycle_ids = [cid for cid in nasa_cycles if 5 <= cid <= 15]

if len(baseline_cycle_ids) < 3:
    raise ValueError("Not enough baseline cycles found (need at least 3 between 5-15).")

nasa_baseline = np.mean(
    [nasa_cycles[cid]['curve'] for cid in baseline_cycle_ids], axis=0
)
print(f"\n[INFO] NASA Golden Baseline built from {len(baseline_cycle_ids)} cycles: "
      f"{baseline_cycle_ids}")

# --- Subtract each cycle from the NASA baseline → difference arrays ---
# The difference curve encodes HOW MUCH the current cycle's shape
# has drifted from healthy. This is the actual model input feature.
X_nasa, y_nasa = [], []

for cycle_id, data in nasa_cycles.items():
    diff_curve = nasa_baseline - data['curve']  # 100-point difference
    X_nasa.append(diff_curve)
    y_nasa.append(data['soh'])

X_nasa = np.array(X_nasa).reshape(-1, 100, 1)  # Shape: (N, 100, 1)
y_nasa = np.array(y_nasa)

# --- Scale SOH labels to [0, 1] for stable training ---
soh_scaler = MinMaxScaler()
y_nasa_scaled = soh_scaler.fit_transform(y_nasa.reshape(-1, 1)).flatten()

# --- Train/validation split ---
X_train, X_val, y_train, y_val = train_test_split(
    X_nasa, y_nasa_scaled, test_size=0.2, random_state=42
)

print(f"\n[INFO] NASA Training set: {X_train.shape}, Validation set: {X_val.shape}")

  [SKIP] Cycle 218: does not cover 3.90V-3.80V window.
  [SKIP] Cycle 225: does not cover 3.90V-3.80V window.
  [SKIP] Cycle 227: does not cover 3.90V-3.80V window.
  [SKIP] Cycle 229: does not cover 3.90V-3.80V window.
  [SKIP] Cycle 249: does not cover 3.90V-3.80V window.
  [SKIP] Cycle 251: does not cover 3.90V-3.80V window.
  [SKIP] Cycle 253: does not cover 3.90V-3.80V window.
  [SKIP] Cycle 265: does not cover 3.90V-3.80V window.
  [SKIP] Cycle 288: does not cover 3.90V-3.80V window.
  [SKIP] Cycle 317: does not cover 3.90V-3.80V window.

[INFO] Successfully processed 249 cycles.

[INFO] NASA Golden Baseline built from 8 cycles: [6, 7, 8, 10, 11, 12, 14, 15]

[INFO] NASA Training set: (199, 100, 1), Validation set: (50, 100, 1)


In [ ]:
# Cell 4: 1D-CNN Base Model & NASA Pre-Training

def build_base_model(input_shape=(100, 1)):
    """
    Builds the 1D-CNN base model for SOH estimation.
    The Conv1D layers act as the 'feature extractor' backbone.
    The Dense layers act as the 'regression head'.
    During transfer learning, only the head will be retrained.
    """
    model = models.Sequential([

        # --- Feature Extractor Backbone (will be frozen during fine-tuning) ---
        layers.Conv1D(filters=32, kernel_size=5, activation='relu',
                      input_shape=input_shape, name='conv1d_1'),
        layers.MaxPooling1D(pool_size=2, name='maxpool_1'),

        layers.Conv1D(filters=64, kernel_size=3, activation='relu', name='conv1d_2'),
        layers.MaxPooling1D(pool_size=2, name='maxpool_2'),

        layers.Flatten(name='flatten'),

        # --- Regression Head (will be replaced during fine-tuning) ---
        layers.Dense(64, activation='relu', name='dense_head'),
        layers.Dense(1, activation='linear', name='output')
    ])
    return model

# --- Build and compile ---
base_model = build_base_model()
base_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='mse',
    metrics=['mae']
)
base_model.summary()

# --- Train on NASA data ---
early_stop_nasa = EarlyStopping(
    monitor='val_loss', patience=10, restore_best_weights=True
)

history_nasa = base_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=16,
    callbacks=[early_stop_nasa],
    verbose=1
)

print("\n[INFO] NASA pre-training complete.")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_1 (Conv1D)               │ (None, 96, 32)         │           192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ maxpool_1 (MaxPooling1D)        │ (None, 48, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 46, 64)         │         6,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ maxpool_2 (MaxPooling1D)        │ (None, 23, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1472)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_head (Dense)              │ (None, 64)             │        94,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 100,737 (393.50 KB)

 Trainable params: 100,737 (393.50 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 50ms/step - loss: 0.1947 - mae: 0.3611 - val_loss: 0.0922 - val_mae: 0.2699
Epoch 2/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 0.0898 - mae: 0.2544 - val_loss: 0.0857 - val_mae: 0.2616
Epoch 3/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0737 - mae: 0.2366 - val_loss: 0.0820 - val_mae: 0.2500
Epoch 4/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0665 - mae: 0.2233 - val_loss: 0.0798 - val_mae: 0.2507
Epoch 5/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.0642 - mae: 0.2175 - val_loss: 0.0690 - val_mae: 0.2356
Epoch 6/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0570 - mae: 0.2056 - val_loss: 0.0690 - val_mae: 0.2205
Epoch 7/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0522 - mae: 0.1962 - val_loss: 0.0560 - val_mae: 0.1971
Epoch 8/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0378 - mae: 0.1682 - val_loss: 0.0379 - val_mae: 0.1663
Epoch 9/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - lo

In [ ]:
# Fix physical dataframe before processing
physical_df = pd.read_csv('/content/drive/MyDrive/battery_combined_data/battery data/BMS2_log_prepared.csv')

# Add type column — negative current means discharge in your setup
physical_df['type'] = physical_df['current_a'].apply(
    lambda x: 'discharge' if x < 0 else 'charge'
)

print("Columns now:", physical_df.columns.tolist())
print("Cycle IDs:", sorted(physical_df['cycle_id'].unique()))
print("Type counts:\n", physical_df['type'].value_counts())
print("Voltage range:", physical_df['voltage_v'].min(),
      "to", physical_df['voltage_v'].max())

Columns now: ['timestamp', 'time_s', 'voltage_v', 'current_a', 'temperature_c', 'battery_id', 'cycle_id', 'power_w', 'status', 'type']
Cycle IDs: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
Type counts:
 type
discharge    4401
Name: count, dtype: int64
Voltage range: 2.84625 to 4.05125


In [ ]:
# Cell 5: Physical Cell Data Processing (Domain B)

# --- Load physical cell dataset ---
Physical_Data = '/content/drive/MyDrive/battery_combined_data/battery data/BMS2_log_prepared.csv'
physical_df = pd.read_csv(Physical_Data)
print("Physical dataset loaded. Shape:", physical_df.shape)
print(physical_df.head())

# --- Add type column based on current sign ---
# No 'type' column exists in this dataset.
# Negative current = discharge, positive = charge.
physical_df['type'] = physical_df['current_a'].apply(
    lambda x: 'discharge' if x < 0 else 'charge'
)

# --- Add 'true_soh' column to physical_df ---
# The process_and_normalize function expects a 'true_soh' column.
# As it's not explicitly present in this dataset, we'll simulate a decreasing SOH
# based on cycle_id for demonstration purposes. In a real application, these
# would be measured SOH values for each cycle.
# Assuming nominal capacity is 2.5 Ah and a slight degradation per cycle.
physical_df['true_soh'] = physical_df['cycle_id'].apply(lambda x: 2.5 - (x - 1) * 0.05)

print("\nType counts:\n", physical_df['type'].value_counts())
print("Voltage range:", physical_df['voltage_v'].min(),
      "to", physical_df['voltage_v'].max())
print("Cycle IDs:", sorted(physical_df['cycle_id'].unique()))

# --- Process through the same universal pipeline ---
physical_cycles = process_and_normalize(physical_df, nominal_capacity=2.5)

# --- Build Physical Golden Baseline from cycles 3-5 ---
# You only have 5 cycles so we use 3-5 instead of 3-8.
# Skipping cycles 1-2 avoids initialization noise.
physical_baseline_ids = [cid for cid in physical_cycles if 3 <= cid <= 5]

if len(physical_baseline_ids) < 2:
    raise ValueError("Not enough physical baseline cycles found (need at least 2 between 3-5).")

physical_baseline = np.mean(
    [physical_cycles[cid]['curve'] for cid in physical_baseline_ids], axis=0
)
print(f"\n[INFO] Physical Golden Baseline built from cycles: {physical_baseline_ids}")

# --- Generate physical difference arrays ---
X_physical, y_physical = [], []

for cycle_id, data in physical_cycles.items():
    diff_curve = physical_baseline - data['curve']
    X_physical.append(diff_curve)
    y_physical.append(data['soh'])

X_physical = np.array(X_physical).reshape(-1, 100, 1)
y_physical  = np.array(y_physical)

# --- Scale SOH using the SAME scaler fitted on NASA data ---
y_physical_scaled = soh_scaler.transform(y_physical.reshape(-1, 1)).flatten()

# --- Train/val split ---
# With only 5 cycles, we use 1 cycle for validation and 4 for training.
X_p_train, X_p_val, y_p_train, y_p_val = train_test_split(
    X_physical, y_physical_scaled, test_size=0.2, random_state=42
)

print(f"\n[INFO] Physical Training set: {X_p_train.shape}, Validation: {X_p_val.shape}")

Physical dataset loaded. Shape: (4401, 9)
             timestamp  time_s  voltage_v  current_a  temperature_c  \
0  2026-05-13 16:36:28     0.0    3.99625     -0.227          26.74   
1  2026-05-13 16:37:29    61.0    3.99500     -0.227          26.59   
2  2026-05-13 16:38:29   121.0    3.99500     -0.227          26.54   
3  2026-05-13 16:39:30   182.0    3.99250     -0.227          26.46   
4  2026-05-13 16:40:31   243.0    3.99125     -0.227          26.49   

  battery_id  cycle_id  power_w   status  
0       BMS2         1      0.9  RUNNING  
1       BMS2         1      0.9  RUNNING  
2       BMS2         1      0.9  RUNNING  
3       BMS2         1      0.9  RUNNING  
4       BMS2         1      0.9  RUNNING  

Type counts:
 type
discharge    4401
Name: count, dtype: int64
Voltage range: 2.84625 to 4.05125
Cycle IDs: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
  [SKIP] Cycle 5: insufficient data points (0).

[INFO] Successfully processed 4 cycles.

[INFO] P

In [ ]:
# Cell 6: Transfer Learning — Freeze Backbone, Replace & Fine-Tune Head

# --- Step 1: Freeze all Conv1D layers (in the original base_model instance) ---
# This ensures that when base_model's layers are used, their trainability is set.
for layer in base_model.layers:
    if 'conv1d' in layer.name:
        layer.trainable = False
        print(f"  [FROZEN] {layer.name}")
    else:
        layer.trainable = True

# --- Step 2: Build fine-tuned model by replacing the Dense head ---

# Define a new input layer for the fine-tuned model, matching the base_model's input shape
new_input = tf.keras.Input(shape=base_model.input_shape[1:], name='transfer_input')

# Manually pass the new_input through the backbone layers of the original base_model
x = new_input
for layer in base_model.layers:
    # Apply layers until the 'flatten' layer (inclusive) to form the backbone
    if layer.name != 'dense_head' and layer.name != 'output':
        x = layer(x)
    else:
        break # Stop after flatten, as these are the head layers to be replaced

# New regularized head — designed for small-data fine-tuning
x = layers.Dense(
    32, activation='relu',
    kernel_regularizer=regularizers.L2(0.01),
    name='ft_dense_1'
)(x)
x = layers.Dropout(0.3, name='ft_dropout')(x)
output = layers.Dense(1, activation='linear', name='ft_output')(x)

fine_tuned_model = tf.keras.Model(
    inputs=new_input,
    outputs=output,
    name='fine_tuned_soh_estimator'
)


fine_tuned_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='mse',
    metrics=['mae']
)

fine_tuned_model.summary()


history_ft = fine_tuned_model.fit(
    X_p_train, y_p_train,
    validation_data=(X_p_val, y_p_val),
    epochs=100, # Reduced epochs for faster execution and to avoid potential issues
    batch_size=1,       # Batch size 1 for small dataset to avoid batching issues with `batch_size=4` on 3 samples
    callbacks=[], # Removed early stopping for debugging constant validation loss
    verbose=1
)

print("\n[INFO] Fine-tuning complete.")

  [FROZEN] conv1d_1
  [FROZEN] conv1d_2


Model: "fine_tuned_soh_estimator"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ transfer_input (InputLayer)     │ (None, 100, 1)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 96, 32)         │           192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ maxpool_1 (MaxPooling1D)        │ (None, 48, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 46, 64)         │         6,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ maxpool_2 (MaxPooling1D)        │ (None, 23, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1472)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ ft_dense_1 (Dense)              │ (None, 32)             │        47,136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ ft_dropout (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ ft_output (Dense)               │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 53,569 (209.25 KB)

 Trainable params: 47,169 (184.25 KB)

 Non-trainable params: 6,400 (25.00 KB)

Epoch 1/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 133ms/step - loss: 4.7140 - mae: 2.0215 - val_loss: 4.7414 - val_mae: 2.0294
Epoch 2/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - loss: 4.4035 - mae: 1.9447 - val_loss: 4.6131 - val_mae: 1.9989
Epoch 3/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - loss: 4.4134 - mae: 1.9483 - val_loss: 4.4872 - val_mae: 1.9685
Epoch 4/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - loss: 4.2850 - mae: 1.9167 - val_loss: 4.3621 - val_mae: 1.9378
Epoch 5/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - loss: 3.9654 - mae: 1.8319 - val_loss: 4.2358 - val_mae: 1.9064
Epoch 6/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - loss: 4.0017 - mae: 1.8429 - val_loss: 4.1174 - val_mae: 1.8764
Epoch 7/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - loss: 3.9396 - mae: 1.8285 - val_loss: 4.0032 - val_mae: 1.8471
Epoch 8/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - loss: 3.5692 - mae: 1.7258 - val_loss: 3.8912 - val_mae: 1.8180
Epoch 9/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - loss: 3.6610 - mae:

In [ ]:
print("NASA SOH min:", y_nasa.min())
print("NASA SOH max:", y_nasa.max())

NASA SOH min: 1.28745252213794
NASA SOH max: 1.85648742081816


In [ ]:
# Cell 7: Hardware-Safe Edge Export — Saving directly to Google Drive

import os

# --- Define save path in your Google Drive ---
SAVE_DIR = '/content/drive/MyDrive/battery_combined_data/battery data/model_export'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"[INFO] Saving all artefacts to: {SAVE_DIR}")

# --- Save deployment config ---
deployment_config = {
    'physical_baseline' : physical_baseline,
    'nominal_capacity'  : 2.5,
    'voltage_grid'      : VOLTAGE_GRID,
    'voltage_low'       : 3.80,
    'voltage_high'      : 3.90
}
joblib.dump(deployment_config, f'{SAVE_DIR}/deployment_config.pkl')
print("[SAVED] deployment_config.pkl")

# --- Save SOH scaler ---
joblib.dump(soh_scaler, f'{SAVE_DIR}/soh_scaler.pkl')
print("[SAVED] soh_scaler.pkl")

# --- Save model architecture ---
with open(f'{SAVE_DIR}/model_architecture.json', 'w') as f:
    f.write(fine_tuned_model.to_json())
print("[SAVED] model_architecture.json")

# --- Save model weights ---
fine_tuned_model.save_weights(f'{SAVE_DIR}/soh_estimator.weights.h5')
print("[SAVED] soh_estimator.weights.h5")

print("\n[DONE] All deployment artefacts saved to Google Drive.")
print(f"Location: {SAVE_DIR}")
print("Files saved:")
print("  - deployment_config.pkl")
print("  - soh_scaler.pkl")
print("  - model_architecture.json")
print("  - soh_estimator.weights.h5")

# --- Verify files are actually there ---
print("\n[VERIFY] Files in export folder:")
for f in os.listdir(SAVE_DIR):
    print(f"  {f}")

[INFO] Saving all artefacts to: /content/drive/MyDrive/battery_combined_data/battery data/model_export
[SAVED] deployment_config.pkl
[SAVED] soh_scaler.pkl
[SAVED] model_architecture.json
[SAVED] soh_estimator.weights.h5

[DONE] All deployment artefacts saved to Google Drive.
Location: /content/drive/MyDrive/battery_combined_data/battery data/model_export
Files saved:
  - deployment_config.pkl
  - soh_scaler.pkl
  - model_architecture.json
  - soh_estimator.weights.h5

[VERIFY] Files in export folder:
  deployment_config.pkl
  soh_scaler.pkl
  model_architecture.json
  soh_estimator.weights.h5
